In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 01:23:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")


Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [3]:
import pandas as pd

# Dictionary data target cabang — copy-paste dari modul
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Langsung jadikan Spark DataFrame
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))
df_target.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



In [4]:
from pyspark.sql.functions import col

# ===== 1. Baca dari HDFS =====
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv",
    header=True,
    inferSchema=True
)

print("Schema sebelum tambah pendapatan:")
df_transaksi.printSchema()

# ===== 2. Tambah kolom pendapatan =====
df_transaksi = df_transaksi.withColumn(
    "pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

print("\nSchema setelah tambah pendapatan:")
df_transaksi.printSchema()

print("\n5 baris pertama:")
df_transaksi.show(5)

print(f"\nTotal baris: {df_transaksi.count()}")

Schema sebelum tambah pendapatan:
root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)


Schema setelah tambah pendapatan:
root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- pendapatan: integer (nullable = true)


5 baris pertama:
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|    

In [5]:
# ===== A. JOIN & PERBANDINGAN TARGET =====
from pyspark.sql.functions import round as spark_round

# Langkah 1: Agregasi total pendapatan per kota
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Langkah 2: Join dengan df_target
hasil_A = ringkasan_kota.join(df_target, on="kota", how="inner")

# Langkah 3: Hitung pencapaian_persen
hasil_A = hasil_A.withColumn(
    "pencapaian_persen",
    spark_round((col("total_pendapatan") / col("target_bulanan") * 100), 2)
)

# Langkah 4: Urutkan dari pencapaian tertinggi
hasil_A = hasil_A.select(
    "kota", "pic_cabang", "total_pendapatan", 
    "target_bulanan", "pencapaian_persen"
).orderBy(col("pencapaian_persen").desc())

print("=== A. Pencapaian Target per Cabang ===")
hasil_A.show()

=== A. Pencapaian Target per Cabang ===


[Stage 11:=============================>                            (2 + 2) / 4]

+----------+----------+----------------+--------------+-----------------+
|      kota|pic_cabang|total_pendapatan|target_bulanan|pencapaian_persen|
+----------+----------+----------------+--------------+-----------------+
| Purworejo|     Fitri|        45650000|      30000000|           152.17|
|      Solo|      Bayu|        33475000|      40000000|            83.69|
|Yogyakarta|      Joko|        47275000|      60000000|            78.79|
|  Magelang|      Rani|        31650000|      45000000|            70.33|
|  Semarang|      Sari|        38175000|      55000000|            69.41|
+----------+----------+----------------+--------------+-----------------+



In [6]:
# ===== B. WINDOW FUNCTION — KATEGORI TERLARIS PER KOTA =====
from pyspark.sql.functions import sum as spark_sum

# Langkah 1: Agregasi pendapatan per (kota, kategori)
pendapatan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Langkah 2: Definisikan window — partisi per kota, urutkan dari pendapatan tertinggi
window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

# Langkah 3: Beri nomor urut dengan row_number()
df_ranked = pendapatan_kategori.withColumn(
    "peringkat", row_number().over(window_kota)
)

# Langkah 4: Ambil hanya peringkat 1 (top-1 per kota)
hasil_B = df_ranked.filter(col("peringkat") == 1) \
    .select("kota", "kategori", "total_pendapatan", "peringkat") \
    .orderBy("kota")

print("=== B. Kategori Terlaris per Kota ===")
hasil_B.show()

=== B. Kategori Terlaris per Kota ===
+----------+--------------------+----------------+---------+
|      kota|            kategori|total_pendapatan|peringkat|
+----------+--------------------+----------------+---------+
|  Magelang|Kesehatan & Kecan...|         7275000|        1|
| Purworejo|Kesehatan & Kecan...|        10075000|        1|
|  Semarang|        Rumah Tangga|        11125000|        1|
|      Solo|Kesehatan & Kecan...|         8425000|        1|
|Yogyakarta|             Fashion|        13325000|        1|
+----------+--------------------+----------------+---------+



In [7]:
# ===== C. SPARK SQL =====

# Daftarkan sebagai temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Kueri SQL murni (bukan DataFrame API)
hasil_C = spark.sql('''
    SELECT 
        t.kota,
        g.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target g ON t.kota = g.kota
    GROUP BY t.kota, g.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')

print("=== C. Jumlah Transaksi per Kota (Spark SQL) ===")
hasil_C.show()

=== C. Jumlah Transaksi per Kota (Spark SQL) ===


[Stage 19:>                                                         (0 + 4) / 4]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



##### D.
Berdasarkan hasil bagian A (pencapaian target) dan B (kategori terlaris per kota), performa lima
cabang menunjukkan pola yang cukup beragam. Cabang Purworejo (PIC: Fitri) tercatat sebagai cabang
berkinerja paling baik, dengan pencapaian target sebesar ±152% meskipun target bulanannya paling
kecil (Rp30.000.000) — kategori penyumbang terbesarnya adalah Kesehatan & Kecantikan. Sebaliknya,
cabang Semarang (PIC: Sari) adalah cabang yang paling perlu mendapat perhatian manajemen, karena
pencapaian targetnya hanya ±69%, angka terendah di antara seluruh cabang, walaupun kategori Rumah
Tangga di kota ini sebenarnya cukup tinggi. Cabang Magelang (±70%) juga berada di posisi rendah
dan perlu dipantau. Menariknya, Yogyakarta memiliki total pendapatan absolut tertinggi (kategori
terlaris: Fashion), namun karena targetnya juga paling besar (Rp60.000.000), pencapaian
persentasenya justru menengah (±79%) — menunjukkan bahwa volume penjualan tinggi tidak selalu
berarti kinerja terbaik jika dibandingkan relatif terhadap target. Rekomendasi bagi manajemen
adalah mengevaluasi strategi penjualan kategori non-unggulan di Semarang dan Magelang, sekaligus
mempelajari faktor keberhasilan Purworejo untuk direplikasi di cabang lain.